# Python Source Map: Discussion Comment Ingestion

This notebook does three things, in order:

1. **Connect** to MongoDB and load GitHub/Discourse API credentials from `config.json`.
2. **Upsert** the hand-maintained Excel source map into the `consensus.python_source_map`
   collection. Since the spreadsheet is periodically edited manually, this cell is safe
   to re-run any time — it keys on the `Episode #` column, so re-running it just updates
   existing rows rather than duplicating them.
3. **Pull every comment** on each row's `Discussion` link — a discuss.python.org topic
   or a github.com issue/PR, detected automatically from the URL — and upsert each
   comment as its own document into `consensus.python_source_map_comments`, linked back
   to its `Episode #`.

Re-running the whole notebook is always safe: every write is an upsert keyed on a
stable identifier (`Episode #` for source rows, a composite `comment_id` + `Episode #` for comments).


## Cell 1: Configuration and MongoDB connection

In [ ]:
import json
import time
import re
import requests
from pymongo import MongoClient, UpdateOne

with open("config.json") as f:
    config = json.load(f)

GITHUB_TOKEN = config.get("github_token")
mongo_uri = config["mongo_uri"]

HEADERS_GH = {
    "Accept": "application/vnd.github.v3+json",
    **({"Authorization": f"token {GITHUB_TOKEN}"} if GITHUB_TOKEN else {}),
}
HEADERS_DISCOURSE = {
    "Accept": "application/json",
    "User-Agent": "PythonSourceMapCommentScraper/1.0",
}

client = MongoClient(mongo_uri)
db = client["consensus"]
source_map_coll = db["python_source_map"]
comments_coll = db["python_source_map_comments"]

print("Connected to MongoDB database 'consensus'.")
print(f"  - {source_map_coll.name}: {source_map_coll.estimated_document_count()} existing documents")
print(f"  - {comments_coll.name}: {comments_coll.estimated_document_count()} existing documents")


## Cell 2: Upsert the Excel source map into `consensus.python_source_map`

Reads the spreadsheet fresh each time and upserts every row by `Episode #`, so this
reflects whatever the sheet currently says without needing to drop/recreate the
collection. Update `EXCEL_PATH` below if the file lives somewhere else.


In [ ]:
import math
import pandas as pd

EXCEL_PATH = "typing_council_and_pep_submissions_combined_clean.xlsx"
KEY_FIELD = "Episode #"  # unique per row in the source spreadsheet

df = pd.read_excel(EXCEL_PATH)

if KEY_FIELD not in df.columns:
    raise ValueError(
        f"Expected a unique '{KEY_FIELD}' column in {EXCEL_PATH} to use as the "
        f"upsert key, but it wasn't found. Columns present: {list(df.columns)}"
    )
if df[KEY_FIELD].duplicated().any():
    dupes = df[df[KEY_FIELD].duplicated(keep=False)][KEY_FIELD].tolist()
    raise ValueError(f"'{KEY_FIELD}' must be unique to upsert safely; duplicates found: {dupes}")


def clean_record(record):
    """Replace pandas/NumPy NaN/NaT with None so PyMongo can serialize it,
    and convert numpy scalar types to plain Python types."""
    cleaned = {}
    for k, v in record.items():
        if isinstance(v, float) and math.isnan(v):
            cleaned[k] = None
        elif pd.isna(v):
            cleaned[k] = None
        else:
            cleaned[k] = v.item() if hasattr(v, "item") else v
    return cleaned


records = [clean_record(r) for r in df.to_dict(orient="records")]

ops = [
    UpdateOne({KEY_FIELD: rec[KEY_FIELD]}, {"$set": rec}, upsert=True)
    for rec in records
]

if ops:
    result = source_map_coll.bulk_write(ops)
    print(
        f"Upserted {len(ops)} rows from {EXCEL_PATH} into "
        f"'{source_map_coll.database.name}.{source_map_coll.name}': "
        f"{result.upserted_count} inserted, {result.modified_count} updated."
    )
else:
    print("No rows found in the spreadsheet.")

# Ensure re-running this cell is always safe/idempotent
source_map_coll.create_index(KEY_FIELD, unique=True)


## Cell 3: Helper functions for pulling comments

- **Discourse** (`discuss.python.org`): fetches the topic's `.json` API, which returns
  only the first ~20 posts inline (`post_stream.posts`) plus the *full* ordered list of
  every post id in the topic (`post_stream.stream`); any ids missing from the inline
  page are batch-fetched via `/t/<id>/posts.json?post_ids[]=...`. This gets every reply
  in the thread, not just the first page. Handles both the normal
  `/t/<slug>/<topic_id>` URL shape and the bare `/t/<topic_id>/<post_number>` shape that
  appears a few times in the sheet, and strips stray trailing punctuation
  (e.g. a trailing `.` or `,`) that crept in from manual spreadsheet editing.
- **Multiple threads per row**: a `Discussion` cell can contain more than one URL -- the discussion for a single proposal sometimes spanned several rounds/threads (e.g. PEP 772's `Discussion` cell has four separate discourse links). `extract_discussion_links` pulls out *every* URL in the cell, in order, regardless of whether they're separated by whitespace, newlines, or commas, and every thread is fetched and stored -- not just the first one found.
- **GitHub** (`github.com/.../issues|pull/N`): fetches the issue/PR body itself (as the
  first "comment") plus every comment on it, paginated at 100/page, using the
  `github_token` from `config.json` if one is provided (otherwise subject to GitHub's
  low anonymous rate limit).
- **Author pseudonymization**: every comment's `author` field is the **unsalted SHA-256 hash** of the platform username (via `hashlib`), computed before the document is ever sent to MongoDB -- the raw username is never stored. The hash is deliberately unsalted so the same person always hashes to the same value, which is what makes it usable as a consistent per-user identifier (e.g. for aggregating voting results) while still pseudonymizing who they are. This is a one-way hash, not encryption: a known or guessable username can still be matched to its hash by anyone who also hashes it, since no secret key is involved.


In [ ]:
import re
import time
import hashlib
import requests

DISCOURSE_TOPIC_RE = re.compile(r"https?://discuss\.python\.org/t/([^/\s\.,]+)/(\d+)")
GITHUB_ISSUE_RE = re.compile(r"https?://github\.com/([^/]+)/([^/]+)/(?:issues|pull)/(\d+)")

TAG_RE = re.compile(r"<[^>]+>")


def clean_url(url):
    """Strip stray trailing punctuation that sometimes creeps into a
    hand-maintained spreadsheet (e.g. a trailing '.' or ',')."""
    return url.strip().rstrip(".,;) ")


def strip_html(html):
    if not html:
        return ""
    text = TAG_RE.sub(" ", html)
    return re.sub(r"\s+", " ", text).strip()


def hash_author(username):
    """Pseudonymize a username with an unsalted SHA-256 hash before it is
    ever written to MongoDB -- the raw username is never stored or
    transmitted to the database. Deliberately unsalted: the same username
    must always hash to the same value, since the hash is used as a
    consistent per-user identifier (e.g. to aggregate voting results across
    rows), and a salt would make that impossible by giving the same person
    a different hash every run.
    """
    if not username:
        return None
    return hashlib.sha256(str(username).strip().encode("utf-8")).hexdigest()


URL_FINDALL_RE = re.compile(r"https?://\S+")


def extract_discussion_links(cell_value):
    """A 'Discussion' cell may contain more than one URL -- a discussion
    that spanned multiple threads (e.g. several rounds of a proposal, or a
    discourse thread plus a related GitHub issue) pasted one after another,
    separated by whitespace/newlines/commas/semicolons. Extract every URL
    in the cell individually rather than assuming there's exactly one.

    Returns a list of (kind, cleaned_url) tuples, in the order they appear
    in the cell. `kind` is 'discourse', 'github', or None (unrecognized).
    """
    text = str(cell_value)
    raw_urls = URL_FINDALL_RE.findall(text)
    results = []
    for raw in raw_urls:
        url = clean_url(raw)
        if not url:
            continue
        if "discuss.python.org" in url:
            results.append(("discourse", url))
        elif "github.com" in url:
            results.append(("github", url))
        else:
            results.append((None, url))
    return results


# ---------------------------------------------------------------------------
# Discourse: fetch every post in a topic
# ---------------------------------------------------------------------------

def _discourse_topic_json_candidates(url):
    """discuss.python.org links appear in two shapes in the sheet:
    the normal /t/<slug>/<topic_id>[/<post_number>] form, and a bare
    /t/<topic_id>/<post_number> form with no slug. Try both."""
    m = DISCOURSE_TOPIC_RE.match(url)
    if not m:
        return []
    first, second = m.groups()
    candidates = [f"https://discuss.python.org/t/{first}/{second}.json"]
    if first.isdigit():
        candidates.append(f"https://discuss.python.org/t/{first}.json")
    return candidates


def fetch_discourse_comments(url, max_retries=3):
    """Fetch every post in a discuss.python.org topic (the first post plus
    every reply), returning a list of comment dicts. Handles topics with
    more posts than the initial page returns by batch-fetching the rest
    via the topic's post_stream ids.
    """
    for json_url in _discourse_topic_json_candidates(url):
        topic_data = None
        for attempt in range(max_retries):
            resp = requests.get(json_url, headers=HEADERS_DISCOURSE, timeout=15)
            if resp.status_code == 200:
                topic_data = resp.json()
                break
            if resp.status_code == 404:
                break  # try the next URL candidate, if any
            if resp.status_code == 429:
                time.sleep(2 ** attempt)
                continue
            print(f"  Discourse returned {resp.status_code} for {json_url}")
            break
        if topic_data is not None:
            break
    else:
        return []

    if topic_data is None:
        return []

    topic_id = topic_data.get("id")
    stream = topic_data.get("post_stream", {}).get("stream", [])
    posts_by_id = {p["id"]: p for p in topic_data.get("post_stream", {}).get("posts", [])}

    missing_ids = [pid for pid in stream if pid not in posts_by_id]
    CHUNK = 50
    for i in range(0, len(missing_ids), CHUNK):
        chunk = missing_ids[i : i + CHUNK]
        params = "&".join(f"post_ids[]={pid}" for pid in chunk)
        batch_url = f"https://discuss.python.org/t/{topic_id}/posts.json?{params}"
        resp = requests.get(batch_url, headers=HEADERS_DISCOURSE, timeout=15)
        if resp.status_code == 200:
            for p in resp.json().get("post_stream", {}).get("posts", []):
                posts_by_id[p["id"]] = p
        time.sleep(0.3)

    comments = []
    for pid in stream:
        p = posts_by_id.get(pid)
        if not p:
            continue
        comments.append({
            "source": "discourse",
            "comment_id": f"discourse:{topic_id}:{p['id']}",
            "topic_id": topic_id,
            "post_id": p["id"],
            "post_number": p.get("post_number"),
            "author": hash_author(p.get("username")),
            "created_at": p.get("created_at"),
            "updated_at": p.get("updated_at"),
            "html": p.get("cooked"),
            "text": strip_html(p.get("cooked")),
            "url": f"https://discuss.python.org/t/{topic_id}/{p.get('post_number')}",
            "discussion_link": url,
        })
    return comments


# ---------------------------------------------------------------------------
# GitHub: fetch the issue/PR body plus every comment on it
# ---------------------------------------------------------------------------

def fetch_github_comments(url, max_retries=3):
    m = GITHUB_ISSUE_RE.search(clean_url(str(url)))
    if not m:
        return []
    owner, repo, number = m.groups()

    comments = []

    # The issue/PR body itself counts as the first "comment" in the thread
    issue_api_url = f"https://api.github.com/repos/{owner}/{repo}/issues/{number}"
    resp = requests.get(issue_api_url, headers=HEADERS_GH, timeout=15)
    if resp.status_code == 200:
        issue = resp.json()
        comments.append({
            "source": "github",
            "comment_id": f"github:{owner}/{repo}:issue:{issue['id']}",
            "repo": f"{owner}/{repo}",
            "issue_number": int(number),
            "author": hash_author((issue.get("user") or {}).get("login")),
            "created_at": issue.get("created_at"),
            "updated_at": issue.get("updated_at"),
            "html": None,
            "text": issue.get("body") or "",
            "url": issue.get("html_url"),
            "discussion_link": url,
        })
    elif resp.status_code == 403:
        print(f"  Rate limit hit fetching {issue_api_url}; provide a github_token in config.json.")
        return comments
    else:
        print(f"  GitHub returned {resp.status_code} for {issue_api_url}")

    page = 1
    while True:
        comments_api_url = (
            f"https://api.github.com/repos/{owner}/{repo}/issues/{number}/comments"
            f"?per_page=100&page={page}"
        )
        resp = None
        for attempt in range(max_retries):
            resp = requests.get(comments_api_url, headers=HEADERS_GH, timeout=15)
            if resp.status_code == 200:
                break
            if resp.status_code == 403:
                print(f"  Rate limit hit fetching {comments_api_url}; provide a github_token in config.json.")
                return comments
            time.sleep(2 ** attempt)
        if resp is None or resp.status_code != 200:
            break

        page_comments = resp.json()
        if not page_comments:
            break

        for c in page_comments:
            comments.append({
                "source": "github",
                "comment_id": f"github:{owner}/{repo}:comment:{c['id']}",
                "repo": f"{owner}/{repo}",
                "issue_number": int(number),
                "author": hash_author((c.get("user") or {}).get("login")),
                "created_at": c.get("created_at"),
                "updated_at": c.get("updated_at"),
                "html": None,
                "text": c.get("body") or "",
                "url": c.get("html_url"),
                "discussion_link": url,
            })

        if len(page_comments) < 100:
            break
        page += 1
        time.sleep(0.2)

    return comments


## Cell 4: Pull comments for every row and upsert them

Reads `Discussion` links **from the `python_source_map` collection**, not from the
Excel file directly, per spec. Each comment is stored as its own document with a
composite key (`comment_id` + `Episode #`) so shared discussion threads across
multiple episodes retain their respective episode associations without overwriting
each other. Each comment also records `thread_index`/`thread_count` so you can tell
which of a row's (possibly several) discussion threads it came from.


In [ ]:
DISCUSSION_FIELD = "Discussion"
EPISODE_FIELD = "Episode #"

source_docs = list(source_map_coll.find({}, {EPISODE_FIELD: 1, DISCUSSION_FIELD: 1}))
print(f"Read {len(source_docs)} rows from '{source_map_coll.database.name}.{source_map_coll.name}'.")

total_upserted = 0
total_comments = 0
skipped_rows = 0
multi_thread_rows = 0

for doc in source_docs:
    episode = doc.get(EPISODE_FIELD)
    cell_value = doc.get(DISCUSSION_FIELD)

    if not cell_value or not isinstance(cell_value, str) or not cell_value.strip():
        skipped_rows += 1
        continue

    # A single "Discussion" cell can contain more than one URL -- the
    # discussion sometimes spanned multiple threads (e.g. several rounds of
    # the same proposal, or a discourse thread plus a related GitHub issue).
    # Fetch comments from every thread found in the cell, not just the first.
    links = extract_discussion_links(cell_value)
    if not links:
        print(f"Episode {episode}: no URL found in Discussion cell, skipping: {cell_value!r}")
        skipped_rows += 1
        continue

    if len(links) > 1:
        multi_thread_rows += 1
        print(f"Episode {episode}: {len(links)} discussion threads found in this row")

    episode_comment_count = 0

    for thread_index, (kind, link) in enumerate(links):
        if kind == "discourse":
            comments = fetch_discourse_comments(link)
        elif kind == "github":
            comments = fetch_github_comments(link)
        else:
            print(f"Episode {episode}: unrecognized discussion link source, skipping: {link}")
            continue

        if not comments:
            print(f"Episode {episode}: no comments retrieved from {link}")
            continue

        ops = []
        for c in comments:
            c[EPISODE_FIELD] = episode
            c["thread_index"] = thread_index  # which of this row's (possibly several) threads this came from
            c["thread_count"] = len(links)
            ops.append(
                UpdateOne(
                    {"comment_id": c["comment_id"], EPISODE_FIELD: episode},
                    {"$set": c},
                    upsert=True,
                )
            )

        result = comments_coll.bulk_write(ops)
        total_upserted += result.upserted_count + result.modified_count
        total_comments += len(comments)
        episode_comment_count += len(comments)

        print(
            f"  Episode {episode} ({kind}, thread {thread_index + 1}/{len(links)}): "
            f"{len(comments)} comment(s) fetched from {link} "
            f"-> {result.upserted_count} inserted, {result.modified_count} updated"
        )
        time.sleep(0.3)  # be polite to the APIs

# Drop single-field unique index if present from prior runs
try:
    comments_coll.drop_index("comment_id_1")
except Exception:
    pass

comments_coll.create_index([("comment_id", 1), (EPISODE_FIELD, 1)], unique=True)
comments_coll.create_index(EPISODE_FIELD)

print()
print(
    f"Done. {total_comments} total comments processed across {len(source_docs) - skipped_rows} rows "
    f"with a recognized discussion link ({multi_thread_rows} of those rows spanned multiple threads; "
    f"{skipped_rows} rows skipped for missing/unrecognized links)."
)


## Notes / limitations

- **Idempotent re-runs**: both collections are written via `upsert`, so re-running any
  cell — or the whole notebook — never creates duplicates. Cell 2 always reflects the
  Excel file's current state; Cell 4 always reflects the latest comments GitHub/Discourse
  have.
- **Rate limits**: GitHub's anonymous API limit is low (60 requests/hour). Set
  `github_token` in `config.json` for any real run — this is required for the handful
  of `github.com` discussion links, and strongly recommended generally.
- **Discourse polling posts**: a `[poll]` post's `cooked` HTML includes the poll widget
  markup; `text` is a stripped-tag rendering, so poll option counts are not cleanly
  separated from prose in the plain-text field. This notebook stores the raw `html` too
  in case a downstream cell wants to parse the poll separately.
- **Deleted/hidden posts**: an occasional post in a long thread may come back from
  Discourse with `cooked: null` (e.g. deleted-by-moderator posts) — these are still
  stored, with `text` as an empty string, rather than silently dropped, so the comment
  count stays consistent with the thread's actual post count.
